# 📘 **Colab Notebook: Context Evaluation Using Llumo**

---

### 📝 **Notebook Overview**

This notebook helps you **evaluate RAG context** using Llumo’s powerful context-level metrics to ensure quality and safety:

✨ **Metrics included:**  
- **Context Utilization** ⚖️  
- **Redundancy Reduction** 🔍  
- **Relevance Retention** ☣️  
- **Hallucination** 🚫  

---

### 🚀 **What you will do in this notebook:**  
1. 📂 Load input data from an Excel file  
2. 🤖 Evaluate the context for context utilization, relevance retention, hallucination etc.
3. 📊 View the detailed evaluation results  

---

🔐 **Note:** The Llumo API key will be securely requested during runtime using Colab’s input prompt.


# ⚙️ **Step 1: Install Dependencies**


In [ ]:
!pip install llumo -q

# 📂 **Step 2: Import Required Libraries**


In [ ]:

import pandas as pd
from llumo import LlumoClient
import getpass
import os


# 🔐 **Step 3: Enter Llumo API Key Securely**


In [ ]:

# Enter your API key securely (won't be visible in output)
os.environ["LLUMO_API_KEY"] = getpass.getpass("Enter your Llumo API key: ")
os.environ["OPEN_API_KEY"] = getpass.getpass("Enter your open API key: ")

# Retrieve the API key
llumo_key = os.environ["LLUMO_API_KEY"]
openai_key = os.environ["OPEN_API_KEY"]

Enter your Llumo API key: ··········
Enter your open API key: ··········


# 🧾 **Step 4: Load the Dataset - with a query column**


In [ ]:

# Make sure 'data.xlsx' is uploaded to your Colab environment
df = pd.read_excel("data.xlsx",usecols = ["query"])

# Preview the data
df.head()


,query
0,How can I return a laptop if I am not satisfie...
1,What do I do if my product arrives with a defect?
2,How does CyberShield handle a data breach?
3,What do I do if my product is missing parts?
4,Can I return a product if I’ve opened the pack...


### 🧠 **Retrieve Context for Each Query**



In [ ]:
import requests
from openai import OpenAI

def get_context(query):
  url = "https://app.llumo.ai/functionCalling/get-context-from-db"
  reqBody = {"query":query}
  response = requests.post(url,json=reqBody)
  return response.json()["contexts"]

df["context"] = get_context(df["query"].to_list())


### 🛠️ **Generate LLM Output Using OpenAI**


In [ ]:

# Ensure 'output' column exists in the DataFrame
df["output"] = ""

# Loop through first 2 rows to generate model response
for indx, row in df.iterrows():
    query = row["query"]
    context = row["context"]

    # Initialize OpenAI client
    client = OpenAI(api_key=openai_key)

    # Call the chat completion API
    response = client.chat.completions.create(
        model="gpt-4",  # Use "gpt-3.5-turbo" if needed
        messages=[
            {
                "role": "user",
                "content": f"Give answer to the given query: {query}, using the given context: {context}."
            }
        ],
        temperature=0.7
    )

    # Extract model output
    llm_output = response.choices[0].message.content

    # Store result in 'output' column
    df.at[indx, "output"] = llm_output



### 📄 **Input DataFrame with Columns — "query", "context", "output"**
Preview the enriched data that will be passed for input evaluation.


In [ ]:
df.head()

,query,output,context
0,How can I return a laptop if I am not satisfie...,ElectraTech accepts returns within 30 days if ...,ElectraTech is your go-to destination for the ...
1,What do I do if my product arrives with a defect?,Contact FutureGadgets customer service. If th...,FutureGadgets is your source for the most adva...
2,How does CyberShield handle a data breach?,CyberShield's response to a data breach involv...,CyberShield Solutions is a premier provider of...
3,What do I do if my product is missing parts?,Contact TechFuture's 24/7 customer support for...,TechFuture is where innovation meets quality. ...
4,Can I return a product if I’ve opened the pack...,No. HiTechHub's return policy requires the it...,HiTechHub is where we offer the latest and mos...


### **Converting the dataframe into a json format: [{},{}]**

In [ ]:
data = df.to_dict(orient='records')

# 🤖 **Step 5: Initialize Llumo Client And Evaluate**
**Context level Evaluation:**
- 🧠 Context Utilization  
- 🔁 Redundancy Reduction  
- 🎯 Relevance Retention  

**Some Other Evauations::**  
- 📝 Input Relevancy  
- ✅ Response Correctness

- 🤖 Hallucination



In [ ]:
# Create an instance of the LlumoClient
client = LlumoClient(llumo_key)

# Evaluate the queries using input-level metrics
resultDf = client.evaluateMultiple(
    data = data,  # DataFrame to evaluate. Must include columns used in prompt_template (e.g., 'query', 'context')
    evals = ["Context Utilization", "Redundancy Reduction", "Relevance Retention", "Response Correctness","Input Relevancy","Hallucination"], # List of metrics
    prompt_template = "Give answer to the query: {{query}}, using context: {{context}}.",  # Prompt pattern used while evaluating. The column names inside {{}} must exist in the DataFrame.This should match the format used during response generation.
    createExperiment = False   # Set to True to save results as an experiment on the Llumo platform. If False, returns results as a DataFrame.

)



======= Running evaluation for: Context Utilization =======

======= Running evaluation for: Redundancy Reduction =======

======= Running evaluation for: Relevance Retention =======

======= Running evaluation for: Response Correctness =======

======= Running evaluation for: Input Relevancy =======

======= Running evaluation for: Hallucination =======


# 📊 **Step 6: View Evaluation Results**


In [ ]:

# Show the results with added metric columns
resultDf.head()


,query,output,context,Context Utilization,Context Utilization Reason,Redundancy Reduction,Redundancy Reduction Reason,Relevance Retention,Relevance Retention Reason,Response Correctness,Response Correctness Reason,Input Relevancy,Input Relevancy Reason,Hallucination,Hallucination Reason
0,How can I return a laptop if I am not satisfie...,ElectraTech accepts returns within 30 days if ...,ElectraTech is your go-to destination for the ...,99,The response accurately reflects the return po...,75,The context has some redundancy. The warranty...,100,The context provides all necessary information...,100,The response accurately reflects the return po...,100,The query directly asks about returning a lapt...,11,The output is a slightly paraphrased version o...
1,What do I do if my product arrives with a defect?,Contact FutureGadgets customer service. If th...,FutureGadgets is your source for the most adva...,100,"The response accurately reflects the warranty,...",81,The context has some redundancy. The warranty...,86,The context provides information about warrant...,99,The response accurately reflects the provided ...,99,"The query directly asks about product defects,...",37,The output mentions a one-year warranty and 30...
2,How does CyberShield handle a data breach?,CyberShield's response to a data breach involv...,CyberShield Solutions is a premier provider of...,100,The response accurately reflects the context b...,71,The context contains some redundancy. Phrases...,85,The context provides a good overview of CyberS...,100,The response accurately reflects CyberShield's...,99,The query directly asks about CyberShield's da...,15,"The output mentions 'contain the breach, mitig..."
3,What do I do if my product is missing parts?,Contact TechFuture's 24/7 customer support for...,TechFuture is where innovation meets quality. ...,81,The response correctly uses the context to ide...,81,The context has some redundancy. The mention ...,31,The context lacks information about handling m...,100,The response accurately addresses the query us...,86,The query directly relates to the context's fo...,36,"The output mentions 24/7 customer support, whi..."
4,Can I return a product if I’ve opened the pack...,No. HiTechHub's return policy requires the it...,HiTechHub is where we offer the latest and mos...,73,The response correctly extracts the return pol...,78,The context has some redundancy. 'HiTechHub' ...,88,The context provides the return policy informa...,99,The response accurately reflects the return po...,100,The query directly addresses the context's ret...,38,The output correctly states the return policy ...
